# Data Exploration and Quality Analysis

This notebook collects the corpus, manually samples some emails, and detects the data quality problems that affect the design of the evaluation. All the results are presented with their figures and the seeds are set so that all samples and counts will be reproducible.

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df = pd.read_parquet("../data/emails.parquet")
print("TOTAL:", len(df))
print(df.groupby(["source_corpus", "label"]).size())

TOTAL: 34930
source_corpus      label
enron              0        14628
nazario            1         8554
phishing_pot       1         5767
spamassassin_ham   0         4302
spamassassin_spam  1         1679
dtype: int64


## Dataset composition

The corpus contains 34,930 emails drawn from four public sources: Enron and SpamAssassin for legitimate mail, and the Nazario and Phishing Pot collections for malicious mail. SpamAssassin contributes both legitimate and malicious, which gives five source labels in total. The classes are close to balanced (roughly 54% legitimate to 46% malicious) with no resampling, and there are two independent sources per class. That last point is what makes the leave-one-corpus-out experiment in Chapter 4 possible.

In [2]:
def show(source, n=15, seed=42, chars=1000):
    sample = df[df.source_corpus == source].sample(n, random_state=seed)
    for i, (_, r) in enumerate(sample.iterrows(), 1):
        print("=" * 78)
        print(f"[{i}/{n}] {source}")
        print("FROM:   ", str(r.from_addr)[:100])
        print("SUBJECT:", str(r.subject)[:100])
        print("DATE:   ", r.date)
        print("-" * 78)
        print(str(r.body_text)[:chars])
        print()

## Manual Review

15 emails sampled from each corpus and the findings are below the samples.

In [3]:
show("nazario")

[1/15] nazario
FROM:    Citizens Bank <businessservice.refcu71543643he.gps@citizensbank.com>
SUBJECT: ***SPAM*** Urgent Message!
DATE:    2007-09-25 00:15:54+07:00
------------------------------------------------------------------------------
Dear business     or     corporate customer   of     Citizens Bank, Citizens      Bank   Customer      Service requests   you     to  complete Money    Manager      GPS Online      Form . This procedure     is      obligatory      for   all Money  Manager      Global      Processing   Solutions™  (GPS) users. Please      click  hyperlink      below  to       access Money     Manager       GPS  Online  Form . http://moneymanagergps-id761864073.citizensbank.com/gps/userdir/onlineform.aspx Please do     not    respond      to  this     email. ********************************************************** ©   Copyright    2007  Citizens    Financial       Group.     All      rights reserved. Y7E6: 0x3, 0x20, 0x3, 0x707, 0x2, 0x98282353, 0x69328126, 0x5, 0

In [4]:
show("phishing_pot")

[1/15] phishing_pot
FROM:    Home Window Experts <hi@soltv>, us
SUBJECT: phishing@pot Your Dream Home Deserves Premium Windows!. Transform Your Home with Stunning New Window
DATE:    2025-06-29 00:44:28+07:00
------------------------------------------------------------------------------
Your Dream Windows - Free Quote See How Much New Windows Could Cost You This Spring Ready for a windows makeover? Get a free, no-obligation quote from a trusted local professional in just a few quick steps. Whether you need window replacements, installations, or something custom, we make it easy to get started. Save time, compare offers, and upgrade your home this spring with Your Dream Windows. Get Your Free Quote If you no longer wish to receive these emails, you can unsubscribe by clicking here This email was sent to you by a business partner. Do you have problems unsubscribing or complain about advertising? click on the link here

[2/15] phishing_pot
FROM:    CITI BANK <aliyumabdullahidamsha@gmail.c

In [5]:
show("enron")

[1/15] enron
FROM:    scott.palmer@enron.com
SUBJECT: prompt post id
DATE:    2001-11-28 05:04:38+07:00
------------------------------------------------------------------------------
Mkt CG - 1418061  

Aruba - 1418057  

Central Transport - 1418060 

1418051  Ontario
 
1418052  Market Mich

1417922   Gulf

1417982  Midcon South

Thanks

[2/15] enron
FROM:    dan.hyvl@enron.com
SUBJECT: RE: Enfolio Contract with CPS
DATE:    2000-11-02 00:20:00+07:00
------------------------------------------------------------------------------
This is being resent because of an addressing error which caused the original 
message to fail.
----- Forwarded by Dan J Hyvl/HOU/ECT on 11/01/2000 05:19 PM -----

	Dan J Hyvl
	11/01/2000 05:17 PM
		
		 To: "Pfister, Christian W." <CWPfister@cps-satx.com>@ENRON
		 cc: "Mc Whirter, Daniel D." <DDMcWhirter, "'James.I.Ducote@enron.com'" 
<James.I.Ducote@enron.com>
		 Subject: RE: Enfolio Contract with CPS

Please review the revised representations and warranties la

In [6]:
show("spamassassin_ham")

[1/15] spamassassin_ham
FROM:    James Tauber <jtauber@jtauber.com>
SUBJECT: Re: A biblical digression
DATE:    2002-08-25 23:50:54+07:00
------------------------------------------------------------------------------
On Sat, 24 Aug 2002 11:07:00 -0700, "John Hall" <johnhall@evergo.net>
said:
> Ran across a site which claimed to explain the original meaning of the
> Ten Commandments.  It seems some of those meanings have evolved a bit,
> too.

By "meanings have evolved" do you [or they] mean that the Hebrew words
have changed meaning or that our understanding of the Hebrew words have
changed? Or do they posit a pre-Mosaic form of the laws that had
evolved by time of the Pentateuch?

> In particular, there was a claim that the commandment on stealing was
> actually specifically about 'man stealing -- selling a free man into
> slavery.

This seems bogus to me. A quick check of the text indicates the the
Hebrew word in question is GANAV which elsewhere in the Pentateuch (eg
Gen 44.8) is us

#### Class tells: what makes an email deceptive

These are the signals a detector should learn.

- Seven of the fifteen samples from Nazario (4, 6, 8, 11, 12, 13, and 15) have warnings to users that their accounts may be suspended, their email inboxes may be disabled or they may lose their files unless they act. There is no similar warning to legitimate senders.
- A mismatch of sender/brand. Examples include Walmart sent by agratandooricuisine.com, PayPal sent by menuwebsites.com, CITI BANK sent by a Gmail address, Netflix.com sent by monkey.dyana.shop. This type of detection would require knowledge of the "From" domain in order to detect it. A body-text based model could not know this.
- Links lead you to a place that is not stated in the message. In example one (Citibank login) the link leads to `74-129-213-82.dhcp.insightbb.com` which is a generic residential broadband IP address instead of being a part of the banks infrastructure. In another example, the link leads to `www.email1.paypal.com.webscraccess.info`. Here the real domain is webscraccess.info while the brand name is simply camouflage.
- Generic greetings. Examples include: "Dear PayPal Member", "ATTENTION DEAR BENEFICIARY!", and "Dear jose@monkey.org" (the collection address was used as a name so there is no actual account associated with this).
- Typical misspellings found in official sounding messages: "indefinitly", "belive", "rightfull", "choise". Also typical of this type of spam is the use of exaggerated language ("hard increase of fraud level") and excessive capitalization (i.e., each word capitalized).
- Evasion aimed at the classifier. HTML injected mid-word to break tokenisation (Pl<SPAN>ease do not re<SPAN>ply, Nazario 14).
- Callback phishing. Both Nazario #3 and #5 (both from 2025) contain links to no malicious URL at all, the payload is a telephone number the user is supposed to dial.

#### Corpus tells: what identifies the source rather than the intent

A classifier that keys on these can score highly while learning nothing about deception.

- Enron carries its own name. "@enron.com" appears in 13 of the 15 From addresses; only samples 3 and 11 are external.
- Enron carries its mail client. Internal forwarding markers and routing codes appear in 5 of 15 (2, 4, 7, 12, 14): `Dan J Hyvl/HOU/ECT`, `Mark E Haedicke/HOU/ECT@ECT`, `Richard B Sanders/HOU/ECT`, `Chris Dorland/CAL/ECT` and `Drew Fossum/ET&S/Enron@ENRON`.
- SpamAssassin ham carries its mailing lists. Footers appear in 10 of 15 (1, 4, 5, 6, 7, 8, 10, 11, 14, 15), naming lists such as ILUG, RPM-List, the SpamAssassin lists and FoRK, with sf.net sponsor adverts appended to some. List tags appear in 10 of 15 subject lines (3, 4, 5, 8, 9, 10, 11, 13, 14, 15): `[SAtalk]`, `[ILUG]`, `[zzzzteana]`, `[SAdev]`.
- SpamAssassin ham carries its anonymisation. Strings such as `yyyy@spamassassin.taint.org` and "zzzzteana", artefacts of this corpus's address masking, appear in no other corpus.
- SpamAssassin ham discusses spam filtering in 4 of 15 (3, 5, 14, 15), so the word "spam" associates with the legitimate class here, a leak running in the opposite direction.
- Language separates the classes. Phishing Pot is largely German, Dutch and Portuguese, while both legitimate corpora are English throughout.

#### Hypotheses, recorded before running the baseline

From the corpus tells above and the temporal separation quantified below, I expect:

- a random stratified split to give F1 above 0.97;
- a cross-corpus split to fall substantially below that;
- a classifier trained to predict source_corpus rather than the label to score close to perfect.

In [7]:
def hits(series, pattern):
    m = series.fillna("").str.contains(pattern, regex=True)
    return int(m.sum()), [i + 1 for i in range(len(m)) if m.iloc[i]]

en = df[df.source_corpus == "enron"].sample(15, random_state=42).reset_index(drop=True)
sa = df[df.source_corpus == "spamassassin_ham"].sample(15, random_state=42).reset_index(drop=True)

print("enron.com senders:", hits(en.from_addr, r"@enron\.com"))
print("Notes artefacts:  ", hits(en.body_text, r"Forwarded by .+?/(?:HOU|CAL|ET&S|NA)"))
print("list tags:        ", hits(sa.subject, r"\[(?:SAtalk|ILUG|zzzzteana|SAdev)\]"))
print("list footers:     ", hits(sa.body_text, r"linux\.ie|freshrpms|lists\.sourceforge|xent\.com/mailman|sf\.net"))

enron.com senders: (13, [1, 2, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15])
Notes artefacts:   (5, [2, 4, 7, 12, 14])
list tags:         (10, [3, 4, 5, 8, 9, 10, 11, 13, 14, 15])
list footers:      (10, [1, 4, 5, 6, 7, 8, 10, 11, 14, 15])


In [8]:
print(df.groupby("source_corpus").date.agg(["min", "max", "count"]))

                                        min                       max  count
source_corpus                                                               
enron             1998-10-30 23:02:00+07:00 2020-12-30 03:53:46+07:00  14628
nazario           1970-01-03 21:43:14+07:00 2038-01-19 10:14:07+07:00   8504
phishing_pot      2019-09-19 02:04:48+07:00 2033-02-25 14:15:52+07:00   5744
spamassassin_ham  2002-01-03 01:55:00+07:00 2028-10-04 23:05:01+07:00   4299
spamassassin_spam 0102-02-02 16:39:51+07:00 2020-10-09 01:01:36+07:00   1674


In [9]:
d = pd.to_datetime(df.date, utc=True, errors="coerce")
valid = (d >= pd.Timestamp("1995-01-01", tz="UTC")) & (d <= pd.Timestamp("2026-07-31", tz="UTC"))

print("Missing or implausible dates:", int((~valid).sum()), f"({100*(~valid).mean():.2f}%)")
print("  missing/unparseable:", int(d.isna().sum()),
      "| out of plausible range:", int((~valid & d.notna()).sum()))
print("\nRate by corpus:")
print((df.assign(bad=~valid).groupby("source_corpus").bad.mean() * 100).round(1))
print("\nPlausible-range date span by corpus:")
print(df[valid].groupby("source_corpus").date.agg(["min", "max", "count"]))

Missing or implausible dates: 165 (0.47%)
  missing/unparseable: 81 | out of plausible range: 84

Rate by corpus:
source_corpus
enron                0.0
nazario              0.7
phishing_pot         0.5
spamassassin_ham     0.1
spamassassin_spam    4.1
Name: bad, dtype: float64

Plausible-range date span by corpus:
                                        min                       max  count
source_corpus                                                               
enron             1998-10-30 23:02:00+07:00 2020-12-30 03:53:46+07:00  14628
nazario           1999-04-08 06:56:34+07:00 2025-12-31 05:30:14+07:00   8490
phishing_pot      2019-09-19 02:04:48+07:00 2026-05-21 18:19:59+07:00   5739
spamassassin_ham  2002-01-03 01:55:00+07:00 2002-12-04 18:54:45+07:00   4298
spamassassin_spam 1997-01-04 07:24:47+07:00 2020-10-09 01:01:36+07:00   1610


## Corpus date analysis

### Date-header reliability

The timestamp data appears to be somewhat inaccurate for 165 out of 34,930 records (0.47%). These inaccuracies may occur when the timestamp is either completely missing or contains an implausible value such as "2038" where 32-bit Unix time overflows or the year 0102. Timestamp accuracy varies significantly among datasets. For instance, while the Enron dataset has zero percentage of inaccurate timestamps, the rates of inaccurate timestamps are higher for other datasets including 0.1 percent for SpamAssassin ham, 0.5 percent for Phishing Pot, 0.7 percent for Nazario and 4.1 percent for SpamAssassin spam.

As might be expected, these inaccuracies tend to track the type of record. Legitimate records generally contain accurate timestamps. Malicious records often include inaccurate or completely absent timestamps. This is consistent with the fact that the headers were likely manipulated by attackers to create those types of records. Therefore, even though a poorly formatted date would provide some weak evidence against a particular record being legitimate or malicious, it could also provide some evidence against a particular record being malicious simply because its format was inconsistent with malicious attacks. The overall impact of this issue on the results is negligible since only about 0.47 percent of the records contained inaccurate or missing timestamp information. Only records containing timestamps outside of the years 1995-2026 were eliminated from consideration in the temporal analysis. The date field is never used as a model feature.

### Temporal confound

After filtering, the date ranges are Enron 1998–2020 and SpamAssassin ham 2002 alone for the legitimate class, and Nazario 1999–2025 and Phishing Pot 2019–2026 for the malicious class.

This creates a couple of issues. First, SpamAssassin ham is limited to a single year (2002), so it should be interpreted as representing a single archive rather than legitimate email in general. Second, Phishing Pot starts in 2019, which is essentially at the very end of the range of legitimate data sets available, resulting in nearly no overlapping period in which recent legitimate and malicious emails would be simultaneously represented.

Therefore, the classes clearly differ not just in terms of content but also by virtue of differing eras. Over two decades, vocabulary, HTML practices and character encoding standards all changed substantially. Thus, if you develop a classifier using this data, it will inevitably be able to differentiate between classes primarily by virtue of the era they represent rather than whether the messages were deceptive or not. A random stratified split cannot detect that, which is what motivates the provenance probe.

In [10]:
mask = df.subject.fillna("").str.contains(r"\*\*\*SPAM\*\*\*", regex=True)
print("Total affected:", int(mask.sum()))
print(df[mask].groupby(["source_corpus", "label"]).size())

Total affected: 76
source_corpus  label
nazario        1        76
dtype: int64


In [11]:
for pat in [r"\*\*\*SPAM\*\*\*", r"\[SPAM\]", r"^SPAM:", r"X-Spam"]:
    m = df.subject.fillna("").str.contains(pat, regex=True)
    print(pat, "->", int(m.sum()), dict(df[m].label.value_counts()))
tags = df.subject.fillna("").str.contains(r"\*\*\*SPAM\*\*\*|\[SPAM\]|^SPAM:", regex=True)
print("distinct malicious subjects with any tag:", int((tags & (df.label == 1)).sum()))

\*\*\*SPAM\*\*\* -> 76 {1: np.int64(76)}
\[SPAM\] -> 18 {1: np.int64(18)}
^SPAM: -> 148 {1: np.int64(148)}
X-Spam -> 7 {0: np.int64(7)}
distinct malicious subjects with any tag: 237


## Spam-filter tags in subject lines

Three inserted spam-filter markers appear in the malicious class and nowhere else: `***SPAM***` on 76 subjects, `SPAM:` on 148, and [SPAM] on 18, which is 242 tag occurrences on 237 distinct subjects, all of them malicious, because a few subjects carry more than one tag. A fourth marker, `X-Spam`, appears on 7 subjects, all of them legitimate. These tags were added by a mail filter rather than by the sender, so left in place they would let a model separate the classes by spotting the filter's own annotation. The three malicious markers are stripped before feature extraction in baseline.py and transformer.py.

In [12]:
LEAK = r"(?:jose@monkey\.org|monkey\.org|phishing@pot|phishing\.pot)"
leaked = (df.body_text.fillna("").str.contains(LEAK, regex=True)
          | df.subject.fillna("").str.contains(LEAK, regex=True))
print("Emails with honeypot addresses:", int(leaked.sum()), f"({100*leaked.mean():.1f}%)")
print(df.assign(leaked=leaked).groupby("source_corpus").leaked.sum())

Emails with honeypot addresses: 3391 (9.7%)
source_corpus
enron                   0
nazario              1497
phishing_pot         1882
spamassassin_ham       12
spamassassin_spam       0
Name: leaked, dtype: int64


In [13]:
pp = df[df.source_corpus == "phishing_pot"]
print("phishing@pot present in Phishing Pot:",
      pp.body_text.fillna("").str.contains("phishing@pot").mean().round(3))

phishing@pot present in Phishing Pot: 0.284


## Collection-address leakage

The honeypot collection address appears in 3,391 emails, 9.7% of the corpus (cell above), almost entirely in the malicious class, and the string phishing@pot alone occurs in 28.4% of Phishing Pot bodies. The address reflects how the corpus was gathered rather than any property of phishing. It is removed before training; Chapter 4 shows that replacing it with a placeholder token merely relocated the leak, so it is replaced with whitespace, and that removing it changes F1 by less than 0.005.

In [14]:
spam_words = r"unsubscribe|abmelden|desinscri|descadastr|afmelden|advertisement|no longer wish"
pp = df[df.source_corpus == "phishing_pot"]
naz = df[df.source_corpus == "nazario"]
print("Phishing Pot with unsubscribe-type footer:",
      pp.body_text.fillna("").str.contains(spam_words, case=False).mean().round(3))
print("Nazario with the same:",
      naz.body_text.fillna("").str.contains(spam_words, case=False).mean().round(3))

Phishing Pot with unsubscribe-type footer: 0.315
Nazario with the same: 0.035


## Label noise in Phishing Pot

Phishing Pot is honeypot-collected mail, everything sent to a trap address, which is unsolicited but not necessarily deceptive. In comparison with 31.5% of its messages carrying a 'list-unsubscribe' style footer (e.g., unsubscribe, abmelden, advertisement etc.), the percentage of messages in Phishing Pot that could be classed as ordinary commercial spam rather than targeted phishing is larger than those contained within Nazario's collection. This is why the task is framed as malicious/unwanted versus legitimate and why the cross-corpus result in Chapter 4 partly measures phishing-to-spam generalisation.

In [15]:
acc = r"[äöüßçãõéèêñ]"
print((df.assign(nonascii=df.body_text.fillna("").str.contains(acc))
         .groupby("source_corpus").nonascii.mean() * 100).round(1))

source_corpus
enron                 0.0
nazario               3.5
phishing_pot         37.9
spamassassin_ham      2.8
spamassassin_spam     1.9
Name: nonascii, dtype: float64


## Language confound

A large share of Phishing Pot is non-English: 37.9% of its messages contain accented characters, against 0% of Enron (the cell above gives the figure per corpus). Although since nearly all legitimate collections of email will be written in English, this allows models to distinguish between legitimate and malicious emails based on language, rather than their purpose. A detection of accented characters is used here to provide a proxy for whether or not an email message is written in a foreign language. It is explicitly noted that the percentages are simply accented character proxies and should not be considered as a complete identification of a given language.

In [16]:
pp = df[df.source_corpus == "phishing_pot"]
b64 = pp.body_text.fillna("").str.contains(r"^[A-Za-z0-9+/=\s]{200,}$", regex=True)
print("Base64-like bodies in Phishing Pot:", int(b64.sum()))

Base64-like bodies in Phishing Pot: 7


## Encoded message bodies

Seven Phishing Pot bodies are almost entirely base64-like text rather than readable content. At this volume the effect is immaterial; the cases are noted and left uncorrected.

## Summary of data-quality issues

| # | Issue | Measure | Handling |
|---|---|---|---|
| 1 | Implausible or missing date headers | 165 (0.47%) | Excluded from temporal split only |
| 2 | Temporal separation between classes | ham 2002 only; Phishing Pot from 2019 | Reported as a limitation |
| 3 | Spam-filter tags in subjects | 242 tags on 237 subjects, all malicious (SPAM 76, SPAM: 148, [SPAM] 18); X-Spam 7 legitimate | Malicious tags stripped before training |
| 4 | Honeypot collection addresses | 3,391 (9.7%) | Replaced with whitespace |
| 5 | Label noise in Phishing Pot | 31.5% list-unsubscribe footers | Task reframed as malicious/unwanted |
| 6 | Language confound | 37.9% of Phishing Pot accented (proxy) | Measured, not corrected |
| 7 | Base64-encoded bodies | 7 | Noted, immaterial |